# Input Format (STimage-1K4M — Legacy ST Sample)

- **image**: H&E-stained histopathology image
    - `{id}.png`: Full resolution tissue slide image
- **coord/**: Spot coordinates
    - `{id}_coord.csv`: Pixel positions of each capture spot on the full resolution image
        - `yaxis`: Pixel row position of the spot centroid (vertical)
        - `xaxis`: Pixel column position of the spot centroid (horizontal)
        - `r`: Radius of the spot in pixels
- **gene_exp/**: Gene expression count matrix
    - `{id}_count.csv`: RNA counts per spot per gene
        - Rows: spot IDs in format `{sample}_{row}x{col}` (e.g. `ST_A1_10x13`)
        - Columns: gene names (e.g. `SAMD11`, `NOC2L`, ...) — up to ~15,000 genes
        - Values: integer RNA read counts (sparse — mostly zeros)

# Target Format

- **wsis/**: H&E-stained whole slide images in pyramidal Generic TIFF (or pyramidal Generic BigTIFF if >4.1GB)
- **st/**: Spatial transcriptomics expressions in a scanpy `.h5ad` object
    - **Observations** (`st.adata.obs`)
        - `in_tissue`: Indicator if the observation is within the tissue (`in_tissue` comes from the initial Visium/Xenium run and might not be accurate, prefer the segmentation obtained by `st.segment_tissue()` instead)
        - `pxl_col_in_fullres`: Pixel column position of the patch/spot centroid in the full resolution image
        - `pxl_row_in_fullres`: Pixel row position of the patch/spot centroid in the full resolution image
        - `array_col`: Patch/spot column position in the array
        - `array_row`: Patch/spot row position in the array
        - `n_counts`: Number of counts for each observation
        - `n_genes_by_counts`: Number of genes detected by counts in each observation
        - `log1p_n_genes_by_counts`: Log-transformed number of genes detected by counts
        - `total_counts`: Total counts per observation
        - `log1p_total_counts`: Log-transformed total counts
        - `pct_counts_in_top_50_genes`: Percentage of counts in the top 50 genes
        - `pct_counts_in_top_100_genes`: Percentage of counts in the top 100 genes
        - `pct_counts_in_top_200_genes`: Percentage of counts in the top 200 genes
        - `pct_counts_in_top_500_genes`: Percentage of counts in the top 500 genes
        - `total_counts_mito`: Total mitochondrial counts per observation *(may not be accurate)*
        - `log1p_total_counts_mito`: Log-transformed total mitochondrial counts *(may not be accurate)*
        - `pct_counts_mito`: Percentage of counts that are mitochondrial *(may not be accurate)*
    - **Variables** (`st.adata.var`)
        - `n_cells_by_counts`: Number of cells detected by counts for each variable
        - `mean_counts`: Mean counts per variable
        - `log1p_mean_counts`: Log-transformed mean counts
        - `pct_dropout_by_counts`: Percentage of dropout events by counts
        - `total_counts`: Total counts per variable
        - `log1p_total_counts`: Log-transformed total counts
        - `mito`: Indicator if the gene is mitochondrial
    - **Unstructured** (`st.adata.uns`)
        - `spatial`: Contains a downscaled version of the full resolution image in `st.adata.uns['spatial']['ST']['images']['downscaled_fullres']`
    - **Observation-wise Multidimensional** (`st.adata.obsm`)
        - `spatial`: Pixel coordinates of spots/patches centroids on the full resolution image (first column = x axis, second column = y axis)
- **metadata/**: Metadata
- **spatial_plots/**: Overlay of the WSI with the ST spots
- **thumbnails/**: Downscaled version of the WSI
- **tissue_seg/**: Tissue segmentation masks
    - `{id}.geojson`: Tissue segmentation mask
    - `{id}_vis.jpg`: Visualization of the tissue mask on the downscaled WSI
- **pixel_size_vis/**: Visualization of the pixel size
- **patches/**: 256×256 H&E patches (0.5µm/px) extracted around ST spots in a `.h5` object optimized for deep learning; each patch is matched to the corresponding ST profile (see **st/**) with a barcode
- **patches_vis/**: Visualization of the mask and patches on a downscaled WSI
- **transcripts/**: Individual transcripts aligned to H&E for Xenium samples; read with `pandas.read_parquet`; aligned coordinates in pixels are in columns `['he_x', 'he_y']`
- **cellvit_seg/**: CellViT nuclei segmentation
- **xenium_seg/**: Xenium segmentation on DAPI aligned to H&E

![VisualONE](https://raw.githubusercontent.com/zeptabot/Hest-Assembling-Pipeline/main/VisualONE.png) ![VisualTWO](https://raw.githubusercontent.com/zeptabot/Hest-Assembling-Pipeline/main/VisualTWO.png)

```
╔══════════════════════════════════════════════════════════════════════════════════╗
║                         STimage → HEST TRANSFORMATION                          ║
╚══════════════════════════════════════════════════════════════════════════════════╝

┌─────────────────────────────────────────────────────────────────────────────────┐
│ INPUTS (STimage)                                                                │
├──────────────────┬──────────────────────────┬──────────────────────────────────┤
│ {id}.png         │ {id}_coord.csv           │ {id}_count.csv                   │
│                  │                          │                                  │
│ H&E image        │ spot_id | xaxis | yaxis  │ spot_id | GENE1 | GENE2 | ...    │
│                  │ 10x13   | 1923  | 2622   │ 10x13   |   0   |   1   | ...    │
│                  │ 10x14   | 1926  | 2844   │ 10x14   |   0   |   0   | ...    │
└──────────────────┴──────────────────────────┴──────────────────────────────────┘
         │                      │                           │
         │           coord.rename(xaxis→X, yaxis→Y)        │
         │                      │                           │
         └──────────────────────┴───────────────────────────┘
                                │
                                ▼
                     STReader().read(img_path,
                       raw_counts_path, spot_coord_path)
                                │
                                ▼
          ┌─────────────────────────────────────────────────┐
          │  HESTData object (st)                           │
          │                                                 │
          │  adata.X  ←  count.csv entire matrix           │
          │                                                 │
          │  adata.obs  (one row per spot):                 │
          │   ├─ pxl_col_in_fullres  ←  coord xaxis        │
          │   ├─ pxl_row_in_fullres  ←  coord yaxis        │
          │   ├─ array_row           ←  "10" from 10x13    │
          │   ├─ array_col           ←  "13" from 10x13    │
          │   ├─ in_tissue           ←  hardcode True      │
          │   │                                             │
          │   │  [sc.pp.calculate_qc_metrics() — AUTO]     │
          │   ├─ n_counts            ←  row.sum()          │
          │   ├─ total_counts        ←  row.sum()          │
          │   ├─ log1p_total_counts  ←  log(total+1)       │
          │   ├─ n_genes_by_counts   ←  (row>0).sum()      │
          │   ├─ log1p_n_genes_by_counts ← log(n_genes+1)  │
          │   ├─ pct_counts_in_top_50_genes                 │
          │   ├─ pct_counts_in_top_100_genes  ← sort row   │
          │   ├─ pct_counts_in_top_200_genes    desc, sum  │
          │   ├─ pct_counts_in_top_500_genes    top N /    │
          │   ├─ total_counts_mito   ← row[MT-*].sum()     │
          │   ├─ log1p_total_counts_mito                    │
          │   └─ pct_counts_mito     ← mito/total×100      │
          │                                                 │
          │  adata.var  (one row per gene):                 │
          │   ├─ mito               ← name.startswith(MT-) │
          │   │                                             │
          │   │  [sc.pp.calculate_qc_metrics() — AUTO]     │
          │   ├─ total_counts       ←  col.sum()           │
          │   ├─ log1p_total_counts ←  log(total+1)        │
          │   ├─ mean_counts        ←  col.mean()          │
          │   ├─ log1p_mean_counts  ←  log(mean+1)         │
          │   ├─ n_cells_by_counts  ←  (col>0).sum()       │
          │   └─ pct_dropout_by_counts ← (col==0)/n×100    │
          │                                                 │
          │  adata.uns['spatial']['ST']['images']           │
          │   └─ downscaled_fullres ← png downscaled       │
          │      [via register_downscale_img() — AUTO]      │
          │                                                 │
          │  adata.obsm['spatial']                          │
          │   └─ [[xaxis, yaxis], ...]  ←  coord CSV       │
          └──────────────────────┬──────────────────────────┘
                                 │
       ┌─────────────────────────┼──────────────────────────────────────┐
       │                         │                                      │
       ▼                         ▼                                      ▼
st.save(                 st.save_spatial_plot()           st.segment_tissue()
  save_img=True,                 │                        st.save_tissue_seg_pkl()
  plot_pxl_size=True)            │                                      │
       │                         │                                      │
       ▼                         ▼                                      ▼
┌─────────────────┐   ┌──────────────────┐               ┌─────────────────────┐
│ wsis/           │   │ spatial_plots/   │               │ tissue_seg/         │
│  .tif           │   │  spots overlaid  │               │  {id}.geojson       │
│                 │   │  on downscaled   │               │  {id}_vis.jpg       │
│ st/             │   │  WSI             │               └─────────────────────┘
│  .h5ad          │   └──────────────────┘
│                 │
│ metadata/       │              ▼
│  metrics.json   │   st.dump_patches(
│                 │     target_patch_size=256,
│ thumbnails/     │     target_pixel_size=0.5)
│  .jpeg          │              │
│                 │     ┌────────┴────────┐
│ pixel_size_vis/ │     ▼                ▼
│  .png           │  ┌──────────┐  ┌─────────────┐
└─────────────────┘  │ patches/ │  │ patches_vis/│
                     │  .h5     │  │  _patch_    │
                     │          │  │  vis.png    │
                     └──────────┘  └─────────────┘

┌──────────────────────────────────────────────┐
│ N/A for STimage (Xenium only — skip)         │
│  transcripts/   cellvit_seg/   xenium_seg/   │
└──────────────────────────────────────────────┘
```

In [12]:
import os
import shutil
import pandas as pd
import tempfile
from huggingface_hub import hf_hub_download

# Define the base paths
base_path = "/Users/bradzap/Developer/GitHub/Hest Assembling Pipeline"

# Define technology folders
tech_folders = {
    "Visium": os.path.join(base_path, "Visum"),
    "VisiumHD": os.path.join(base_path, "VisumHD"),
    "ST": os.path.join(base_path, "ST")
}

# Ensure all technology folders exist
for folder in tech_folders.values():
    os.makedirs(folder, exist_ok=True)

# Function to download a specific slide by name
def download_slide_by_name(slide_name):
    """
    Download a specific slide from STimage-1K4M dataset by its slide name.
    
    Args:
        slide_name (str): The unique slide identifier (e.g., "GSE144239_GSM4284316")
    
    Returns:
        str: Path to the downloaded sample folder
    """
    # Download metadata to find the slide
    print(f"Looking up slide: {slide_name}")
    try:
        # Use temporary directory for metadata download
        with tempfile.TemporaryDirectory() as temp_dir:
            meta_path = hf_hub_download(
                repo_id="jiawennnn/STimage-1K4M",
                filename="meta/meta_all_gene02122025.csv",
                repo_type="dataset",
                local_dir=temp_dir
            )
            
            # Read metadata
            meta_df = pd.read_csv(meta_path)
            
            # Find the specific slide
            slide_row = meta_df[meta_df['slide'] == slide_name]
            
            if slide_row.empty:
                print(f"Error: Slide '{slide_name}' not found in metadata")
                print(f"Available slides: {meta_df['slide'].tolist()[:10]}... (showing first 10)")
                return None
            
            # Get slide information
            slide_row = slide_row.iloc[0]
            technology = slide_row['tech']
            
            print(f"Found slide: {slide_name}")
            print(f"Technology: {technology}")
            print(f"Species: {slide_row['species']}")
            print(f"Tissue: {slide_row['tissue']}")
        
        # Map technology to folder name
        tech_mapping = {
            "Visium": "Visum",
            "VisiumHD": "VisumHD", 
            "ST": "ST"
        }
        
        target_folder = tech_folders.get(technology)
        if not target_folder:
            print(f"Warning: Unknown technology '{technology}', defaulting to ST folder")
            target_folder = tech_folders["ST"]
        
        # Create slide folder with 'input' and 'output' subdirectories
        sample_folder = os.path.join(target_folder, slide_name)
        input_folder = os.path.join(sample_folder, "input")
        output_folder = os.path.join(sample_folder, "output")
        
        os.makedirs(input_folder, exist_ok=True)
        os.makedirs(output_folder, exist_ok=True)
        
        print(f"Target folder: {sample_folder}")
        print(f"Input folder: {input_folder}")
        print(f"Output folder: {output_folder}")
        
        # Use the actual technology folder name from the repository (case-sensitive)
        tech_folder_name = technology  # Keep original case: ST, Visium, VisiumHD
        
        # File naming convention: {slide_id}_{type}.{ext}
        # coord: {slide_id}_coord.csv
        # gene_exp: {slide_id}_count.csv
        # image: {slide_id}.png
        file_mappings = {
            'coord': f"{slide_name}_coord.csv",
            'gene_exp': f"{slide_name}_count.csv",
            'image': f"{slide_name}.png"
        }
        
        for file_type, filename in file_mappings.items():
            try:
                # Construct the path for this specific slide
                dataset_path = f"{tech_folder_name}/{file_type}/{filename}"
                
                print(f"Attempting to download {file_type} for {slide_name}...")
                print(f"Dataset path: {dataset_path}")
                
                try:
                    # Download to temp directory first to avoid subdirectory creation
                    with tempfile.TemporaryDirectory() as temp_dir:
                        temp_path = hf_hub_download(
                            repo_id="jiawennnn/STimage-1K4M",
                            filename=dataset_path,
                            repo_type="dataset",
                            local_dir=temp_dir
                        )
                        
                        # Copy to input folder with just the filename
                        dest_path = os.path.join(input_folder, filename)
                        shutil.copy(temp_path, dest_path)
                        print(f"Successfully downloaded {file_type} to: {dest_path}")
                        
                except Exception as e:
                    print(f"Could not download {dataset_path}: {e}")
                    
            except Exception as e:
                print(f"Error processing {file_type}: {e}")
        
        print(f"\nSample files organized in: {input_folder}")
        print(f"Empty output directory created at: {output_folder}")
        return sample_folder
        
    except Exception as e:
        print(f"Error during download: {e}")
        return None

# Example usage - specify the slide name you want to download
# Replace with your desired slide name from the dataset
slide_to_download = "GSE144239_GSM4284316"  # Example slide name

# Download the specific slide
downloaded_path = download_slide_by_name(slide_to_download)

if downloaded_path:
    print(f"\nSuccessfully downloaded slide to: {downloaded_path}")
    print(f"Input files are in: {os.path.join(downloaded_path, 'input')}")
    print(f"Output directory is at: {os.path.join(downloaded_path, 'output')}")
else:
    print("\nFailed to download slide. Please check the slide name and try again.")

Looking up slide: GSE144239_GSM4284316
Found slide: GSE144239_GSM4284316
Technology: ST
Species: human
Tissue: skin
Target folder: /Users/bradzap/Developer/GitHub/Hest Assembling Pipeline/ST/GSE144239_GSM4284316
Input folder: /Users/bradzap/Developer/GitHub/Hest Assembling Pipeline/ST/GSE144239_GSM4284316/input
Output folder: /Users/bradzap/Developer/GitHub/Hest Assembling Pipeline/ST/GSE144239_GSM4284316/output
Attempting to download coord for GSE144239_GSM4284316...
Dataset path: ST/coord/GSE144239_GSM4284316_coord.csv


GSE144239_GSM4284316_coord.csv: 0.00B [00:00, ?B/s]

Successfully downloaded coord to: /Users/bradzap/Developer/GitHub/Hest Assembling Pipeline/ST/GSE144239_GSM4284316/input/GSE144239_GSM4284316_coord.csv
Attempting to download gene_exp for GSE144239_GSM4284316...
Dataset path: ST/gene_exp/GSE144239_GSM4284316_count.csv


ST/gene_exp/GSE144239_GSM4284316_count.c(…):   0%|          | 0.00/45.9M [00:00<?, ?B/s]

Successfully downloaded gene_exp to: /Users/bradzap/Developer/GitHub/Hest Assembling Pipeline/ST/GSE144239_GSM4284316/input/GSE144239_GSM4284316_count.csv
Attempting to download image for GSE144239_GSM4284316...
Dataset path: ST/image/GSE144239_GSM4284316.png


ST/image/GSE144239_GSM4284316.png:   0%|          | 0.00/11.8M [00:00<?, ?B/s]

Successfully downloaded image to: /Users/bradzap/Developer/GitHub/Hest Assembling Pipeline/ST/GSE144239_GSM4284316/input/GSE144239_GSM4284316.png

Sample files organized in: /Users/bradzap/Developer/GitHub/Hest Assembling Pipeline/ST/GSE144239_GSM4284316/input
Empty output directory created at: /Users/bradzap/Developer/GitHub/Hest Assembling Pipeline/ST/GSE144239_GSM4284316/output

Successfully downloaded slide to: /Users/bradzap/Developer/GitHub/Hest Assembling Pipeline/ST/GSE144239_GSM4284316
Input files are in: /Users/bradzap/Developer/GitHub/Hest Assembling Pipeline/ST/GSE144239_GSM4284316/input
Output directory is at: /Users/bradzap/Developer/GitHub/Hest Assembling Pipeline/ST/GSE144239_GSM4284316/output


In [14]:
import os
import re
import tempfile
import scanpy as sc
import numpy as np
import pandas as pd
from hest import STReader

# ─────────────────────────────────────────────────────────────────
# DERIVE PATHS from previous block's variables
# ─────────────────────────────────────────────────────────────────

SAMPLE_ID  = slide_to_download
input_dir  = os.path.join(downloaded_path, 'input')
output_dir = os.path.join(downloaded_path, 'output')
TECHNOLOGY = os.path.basename(os.path.dirname(downloaded_path))  # ST / Visium / VisiumHD

SPOT_PARAMS = {
    'ST':       {'spot_diameter': 100., 'inter_spot_dist': 200.},
    'Visium':   {'spot_diameter': 55.,  'inter_spot_dist': 100.},
    'VisiumHD': {'spot_diameter': 2.,   'inter_spot_dist': 2.},
}

image_path = os.path.join(input_dir, f'{SAMPLE_ID}.png')
coord_path = os.path.join(input_dir, f'{SAMPLE_ID}_coord.csv')
count_path = os.path.join(input_dir, f'{SAMPLE_ID}_count.csv')

print(f'Sample:     {SAMPLE_ID}')
print(f'Technology: {TECHNOLOGY}')
print(f'Output:     {output_dir}')

# ─────────────────────────────────────────────────────────────────
# STEP 1 — Read CSVs ourselves (both are comma-separated in STimage)
# ─────────────────────────────────────────────────────────────────

counts = pd.read_csv(count_path, index_col=0)   # spots × genes
coord  = pd.read_csv(coord_path, index_col=0)   # spots × (yaxis, xaxis, r)

# Align on common spot IDs
common_idx = counts.index.intersection(coord.index)
counts = counts.loc[common_idx]
coord  = coord.loc[common_idx]

# ─────────────────────────────────────────────────────────────────
# STEP 2 — Strip sample prefix from spot IDs
#   "GSE144239_GSM4284316_10x26" → "10x26"
#   STReader requires this ROWxCOL format for array_row/array_col
# ─────────────────────────────────────────────────────────────────

short_ids = [re.search(r'(\d+x\d+)$', idx).group(1) for idx in counts.index]
counts.index = short_ids
coord.index  = short_ids

# ─────────────────────────────────────────────────────────────────
# STEP 3 — Build AnnData with obsm['spatial'] = [[xaxis, yaxis], ...]
#   col 0 = xaxis → pxl_col_in_fullres
#   col 1 = yaxis → pxl_row_in_fullres
# ─────────────────────────────────────────────────────────────────

adata = sc.AnnData(counts)
adata.obsm['spatial'] = coord[['xaxis', 'yaxis']].values  # shape (n_spots, 2)

# ─────────────────────────────────────────────────────────────────
# STEP 4 — Build HESTData via custom_adata
#   Bypasses broken spot_coord_path branch (missing sc import + sep='\t')
#   STReader still handles: pixel size estimation, register_downscale_img,
#   obs fields (pxl_col/row, array_row/col, in_tissue), index zero-padding
# ─────────────────────────────────────────────────────────────────

st = STReader().read(
    img_path=image_path,
    custom_adata=adata,
    **SPOT_PARAMS[TECHNOLOGY]
)

# ─────────────────────────────────────────────────────────────────
# STEP 5 — Add mito flag and rerun QC for mito obs fields
#   HESTData.__init__() auto-runs calculate_qc_metrics() but without
#   qc_vars=['mito'], so we re-run it here with the mito flag set
# ─────────────────────────────────────────────────────────────────

st.adata.var['mito'] = st.adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(st.adata, qc_vars=['mito'], inplace=True)

# ─────────────────────────────────────────────────────────────────
# STEP 6 — wsis/ + st/ + thumbnails/ + metadata/ + pixel_size_vis/
# ─────────────────────────────────────────────────────────────────

st.save(
    path=output_dir,
    save_img=True,
    pyramidal=True,
    bigtiff=False,
    plot_pxl_size=True
)

# ─────────────────────────────────────────────────────────────────
# STEP 7 — spatial_plots/
# ─────────────────────────────────────────────────────────────────

st.save_spatial_plot(save_path=output_dir)

# ─────────────────────────────────────────────────────────────────
# STEP 8 — tissue_seg/  (swap to method='otsu' if no GPU)
# ─────────────────────────────────────────────────────────────────

st.segment_tissue(method='deep')
st.save_tissue_contours(output_dir, 'tissue_seg')
st.save_tissue_vis(output_dir, 'tissue_seg')

# ─────────────────────────────────────────────────────────────────
# STEP 9 — patches/ + patches_vis/
# ─────────────────────────────────────────────────────────────────

st.dump_patches(
    patch_save_dir=output_dir,
    name='patches',
    target_patch_size=256,
    target_pixel_size=0.5,
    dump_visualization=True
)

print(f'\n✓ Done: {output_dir}')

Sample:     GSE144239_GSM4284316
Technology: ST
Output:     /Users/bradzap/Developer/GitHub/Hest Assembling Pipeline/ST/GSE144239_GSM4284316/output


AttributeError: 'ImageWSI' object has no attribute 'shape'

You can also visualize an overlay of the aligned spots on the downscaled WSI

### When should I provide an alignment file and when should I use autoalignment?

#### Step 1: check if a tissue_positions.csv/tissue_position_list.csv already provides a correct alignment

In most cases, if a `spatial/` folder containing a `tissue_positions.csv` or `tissue_position_list.csv` is available you don't need any autoalignment/alignment file.

Try the following:

`st = VisiumReader().read(fullres_img_path, bc_matric_path, spatial_coord_path=spatial_path)`, where `spatial_path` is contains `tissue_positions.csv` or `tissue_position_list.csv`. You can manually inspect the alignment by saving a visualization plot that takes the full resolution image, downscale it and overlays it with the spots (using `st.save_spatial_plot(save_dir)`). If the alignment is incorrect, try step 2.

#### Step 2: check if a .json alignment file is provided

If a `.json` alignment file is available, try: `VisiumReader().read(fullres_img_path, bc_matric_path, spatial_coord_path=spatial_path, alignment_file_path=align_path)`. You can also omit the `spatial_coord_path` as `VisiumReader().read(fullres_img_path, bc_matric_path, alignment_file_path=align_path)`

#### Step 3: attempt auto-alignment

If at least 3 corner fiducials are not cropped out and are reasonably visible, you can attempt an autoalignment with `VisiumReader().read(fullres_img_path, bc_matric_path`. (if no spatial folder and no alignment_file_path is provided, it will attempt autoalignment by default, you can also force auto-alignment by passing `autoalign='always'`). 

### Examples:

In [ ]:
from hest import VisiumReader

fullres_img_path = 'my_path/image.tif'
bc_matrix_path = 'my_path/filtered_bc_matrix.h5'
spatial_coord_path = 'my_path/spatial'
alignment_file_path = 'my_path/alignment.txt'

st = VisiumReader().read(
    fullres_img_path, # path to a full res image
    bc_matrix_path, # path to filtered_feature_bc_matrix.h5
    spatial_coord_path=spatial_coord_path # path to a space ranger spatial/ folder containing either a tissue_positions.csv or tissue_position_list.csv
)

# if no spatial folder is provided, but you have an alignment file
st = VisiumReader().read(
    fullres_img_path, # path to a full res image
    bc_matrix_path, # path to filtered_feature_bc_matrix.h5
    alignment_file_path=alignment_file_path # path to a .json alignment file
)

# if both the alignment file and the spatial folder are missing, attempt auto-alignment
st = VisiumReader().read(
    fullres_img_path, # path to a full res image
    bc_matrix_path, # path to filtered_feature_bc_matrix.h5
)


alignment file is  None


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'my_path/filtered_bc_matrix.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

### Auto read
Given that `visium_dir` contains a full resolution image and all the necessary Visium files such as the `filtered_bc_matrix.h5` and the `spatial` folder, `VisiumReader.auto_read(path)` should be able to automatically read the sample. Prefer `read` for a more fine grain control.


In [ ]:
from hest import VisiumReader

# Set this to a folder containing a full-res image, filtered matrix and spatial files.
visium_dir = "PATH_TO_VISIUM_DIR"

# attempt autoread
st = VisiumReader().auto_read(visium_dir)

FileNotFoundError: No such file or directory: PATH_TO_VISIUM_DIR

### II. Xenium reader

### Download Xenium sample from 10x genomics website

Download the following xenium files and place them in the same directory

https://www.10xgenomics.com/datasets/human-skin-data-xenium-human-multi-tissue-and-cancer-panel-1-standard

In [ ]:
%%bash

mkdir downloads
cd downloads
curl -O https://cf.10xgenomics.com/samples/xenium/1.9.0/Xenium_V1_hSkin_nondiseased_section_1_FFPE/Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs.zip
unzip Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs.zip -d Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs

python(9119) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
mkdir: downloads: File exists


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 11.0G  100 11.0G    0     0  6416k      0  0:30:00  0:30:00 --:--:-- 3252k38  955k  0:22:37 1774k2:54 1934k3  0:25:13 1342k2:41  0:21:26 19.7M6  0:16:34 3837k  7975k      0  0:24:08  0:05:46  0:18:22 3782k 0:05:48  0:18:19 6025k0:06:14  0:17:40 8030k 8063k      0  0:23:52  0:06:22  0:17:30 8519k:42 2061k53kk0:18:37 1149k  0:18:52  742k13 1295k0:08:57  0:19:23 1669k    0  0:28:53  0:09:10  0:19:43 1299k0:10:40  0:19:08 5885k2189k:18:46 14.1M0:19:30 2312k3.6M30  897k2:30 2794k7 3670k:47  0:10:34 3900k0:10:27 3557k 0:19:45  0:10:26 3223k12  0:19:47  0:10:25 3948k:10:21 4002k0:10:02 2281k1326k:29  0:10:04  945k20:30  0:10:04  818k:32  0:10:22  724k50  0:10:23 2220k0:19 4351k94k:09:12 2989k09:13 1009k39 3869k:23  0:07:41 7491kM6:09  0:06:06 19.6M4:12 5356k  0:04:08 5764k0:02:02 3593k55  0:02:02 3362k4  0:28:15  0:01:59 1025k0:28:17

Archive:  Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs.zip
  inflating: Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs/analysis_summary.html  
  inflating: Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs/cell_boundaries.parquet  
  inflating: Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs/cells.csv.gz  
  inflating: Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs/gene_panel.json  
  inflating: Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs/morphology_focus.ome.tif  
  inflating: Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs/nucleus_boundaries.parquet  
 extracting: Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs/transcripts.zarr.zip  
  inflating: Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs/cell_feature_matrix.h5  
  inflating: Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs/cells.parquet  
  inflating: Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs/morphology_mip.ome.tif  
 extracting: Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs/analysis.zarr.zip  
 extracting: Xenium


/Users/bradzap/Developer/GitHub/HEST/tutorials/downloads/Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs/morphology.ome.tif:  write error (disk full?).  Continue? (y/n/^C) 


CalledProcessError: Command 'b'\nmkdir downloads\ncd downloads\ncurl -O https://cf.10xgenomics.com/samples/xenium/1.9.0/Xenium_V1_hSkin_nondiseased_section_1_FFPE/Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs.zip\nunzip Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs.zip -d Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs\n'' returned non-zero exit status 50.

In [ ]:
%%bash

cd downloads/Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs
wget https://cf.10xgenomics.com/samples/xenium/1.9.0/Xenium_V1_hSkin_nondiseased_section_1_FFPE/Xenium_V1_hSkin_nondiseased_section_1_FFPE_he_image.ome.tif
wget https://cf.10xgenomics.com/samples/xenium/1.9.0/Xenium_V1_hSkin_nondiseased_section_1_FFPE/Xenium_V1_hSkin_nondiseased_section_1_FFPE_he_imagealignment.csv

python(9662) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
bash: line 3: wget: command not found
bash: line 4: wget: command not found


CalledProcessError: Command 'b'\ncd downloads/Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs\nwget https://cf.10xgenomics.com/samples/xenium/1.9.0/Xenium_V1_hSkin_nondiseased_section_1_FFPE/Xenium_V1_hSkin_nondiseased_section_1_FFPE_he_image.ome.tif\nwget https://cf.10xgenomics.com/samples/xenium/1.9.0/Xenium_V1_hSkin_nondiseased_section_1_FFPE/Xenium_V1_hSkin_nondiseased_section_1_FFPE_he_imagealignment.csv\n'' returned non-zero exit status 127.

In [ ]:
from hest import XeniumReader

xenium_folder_path = 'downloads/Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs'

st = XeniumReader().auto_read(xenium_folder_path)

Exception: Couldn't find an image automatically, make sure that the folder downloads/Xenium_V1_hSkin_nondiseased_section_1_FFPE_outs contains an image of one of these types: ['.tif', '.jpg', '.btf', '.png', '.tiff', '.TIF', 'ndpi', 'nd2']

### Working with larger than RAM Xenium samples (Xenium 5k)
We support larger than RAM transcripts pooling powered by dask. Dask will automatically chunk the data such that it never has to hold the entire transcript dataframe in memory.

Dask will attempt to process one partition per thread. To avoid loading large partitions on systems having a low amount of RAM, we advise using a high number of partitions (>100), as well as a single worker and a low number of threads (<4 depending on RAM available).

> Note: feel free to open the dask dashboard to visualize workers, partitions and resources (usually on http://localhost:8787)

In [ ]:
from dask.distributed import LocalCluster, Client

cluster = LocalCluster(
    "127.0.0.1:8786",
    n_workers=1, # increase depending on RAM available
    memory_limit="20GB", # dask will kill the worker if this is exceeded
    threads_per_worker=1, # increase depending on RAM available
)
client = Client(cluster)
print('dashboard is available at: ', client.dashboard_link)

st = XeniumReader().auto_read(
    xenium_folder_path, 
    use_dask=True, 
    nb_partitions=100
)

In [ ]:
st.save_spatial_plot('./')

In [ ]:
st.segment_tissue()

We then compute patches centered around pseudo-visium transcript bins.

> Warning: note that patches might be larger than transcript bins. 

In [ ]:
st.dump_patches('patches', target_patch_size=224, target_pixel_size=0.5)

In [ ]:
st.save('save', save_img=False)

### Adding new samples to HEST
This section explains how to format new samples for HuggingFace

### I. Sample preparation

#### 1. Download raw datasets in structured folders

Create the following folder structure:
```python
my_data/
    xenium/
        dataset_name_1/
            subseries1/
                sample_1/
                    ...
                sample_2/
                    ...
                ...
        dataset_name_2/
            sample_1/
                ...
    visium-hd/
        ...
    visium/
        ...
```

Then, download corresponding files for xenium, visium, visium-hd... See reader examples above or refer to the doc for the list of required files.

#### 2. Add metadata rows to CSV
Fill columns in the sample CSV, specifically: dataset_title, subseries should match with the folder structure.

### II. Sample processing

#### 1. Process raw samples

Set the path to your sample CSV in `tutorials/scripts/1_process_raw_samples.py`, also modify the memory_limit, we recommend setting dask memory limit to at least 20GB for Xenium 5k, eventhough 30GB is safer.

Process and save raw samples by launching `python tutorials/scripts/1_process_raw_samples.py`.

Processing Xenium 5k will take time, you can monitor progress on the dask dashboard (usually at `localhost:8787` after launch).


#### 2. Check Xenium DAPI/HE alignment and realign if necessary

By default, the Xenium platform uses a single affine transform for the whole WSI. For some samples, the resulting alignment might be unsatisfactory.
In order to visualize the affine alignment, either:
- open the resulting `.geojson` files (in `processed/`) in QuPath (might not work with QuPath >=0.6) \
\
or
<br/> 
<br/> 
- launch `tutorials/scripts/2_check_xenium_alignment.py`. 

#### 3. Micro-align DAPI to HE with Valis

If the alignment from steps (1-2) is unsatisfactory, we highly recommend using Valis non-rigid micro-alignment for sub-cellular alignment precision. This is crucial for correctly mapping transcripts and cells to H&E.


##### a. Register DAPI to HE with Valis

We provide a modified Valis version for simplified pythonic use and improved precision, please check-out the original repository [here](https://github.com/MathOnco/valis).

Open [3a_microalign_xenium.py](./scripts/3a_microalign_xenium.py), in this example, we use `morphology_focus_0000.ome.tif` as the DAPI slide, feel free to use `morphology_focus.ome.tif`.

Monitor alignment quality at `results/{sample_name}/{date}/overlaps/`, check both `_rigid_overlap.png` and `micro_reg.png`.


##### b. Warp transcripts, cells and nuclei using Valis and Dask

See [3b_microalign_xenium.py](./scripts/3b_microalign_xenium.py) to warp transcripts, nuclei and cells from DAPI to H&E.

Then re-run step 3.b, in order to compare quality.

#### 4. Segment with CellViT

See [4_segment_cellvit.py](./scripts/4_segment_cellvit.py) to segment with CellViT.


#### 5. Copy processed files to HEST_results/

See [5_copy_processed.py](./scripts/5_copy_processed.py) to copy files to the structure expected by huggingface.


#### 6. Generate a new HEST_vX_Y_Z.csv sheet

See [6_generate_new_meta.py](./scripts/6_generate_new_meta.py) to create a new HEST_vX_Y_Z.csv. Once created, copy it to the `HEST_results/` folder.


#### 7. Upload to HuggingFace

See [7_upload_huggingface.py](./scripts/7_upload_huggingface.py) to upload to HuggingFace via a PR.

